# Somalia Forecast Pipeline

Tests the shared `project.helper.predictions` pipeline for weather, water prices, food/CMB prices, and fuel prices.

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'project').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project.helper.predictions import (
    predict_somalia_weather,
    predict_somalia_water_prices,
    predict_somalia_food_prices,
    predict_global_fuel_price,
    summarize_weather,
)


ImportError: cannot import name 'predict_global_fuel_price' from 'project.helper.predictions' (/home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/helper/predictions.py)

## Test 1: Loaded data and cached forecasts

In [ ]:
weather_cache = predict_somalia_weather(data_source='cache', forecast_source='cache')
water_cache = predict_somalia_water_prices(source='cache')
food_cache = predict_somalia_food_prices(source='cache')
fuel_local = predict_global_fuel_price(source='local')

assert len(weather_cache.history) > 0
assert all(len(frame) > 0 for frame in weather_cache.forecasts.values())
assert len(water_cache.history) > 0 and len(water_cache.forecast) > 0
assert len(food_cache.history) > 0 and len(food_cache.forecast) > 0
assert len(fuel_local.history) > 0 and len(fuel_local.forecast) > 0
assert fuel_local.history['value'].isna().sum() == 0

print('weather rows:', len(weather_cache.history), summarize_weather(weather_cache.history))
print('water rows:', len(water_cache.history), 'forecast rows:', len(water_cache.forecast))
print('food rows:', len(food_cache.history), 'forecast rows:', len(food_cache.forecast))
print('fuel rows:', len(fuel_local.history), 'forecast rows:', len(fuel_local.forecast))
print(fuel_local.forecast.head())


[predictions] Gedo polygon: [(42.8903167, 4.3140417), (43.0996111, 3.0833333), (41.5229722, 1.2651944), (40.9924083, 2.7923778), (42.0586444, 4.1903472), (42.8903167, 4.3140417)]


[predictions] CDS bbox [north, west, south, east]: [4.414041699999999, 40.8924083, 1.1651943999999999, 43.1996111]


[predictions] Loaded cached weather data: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/gedo_monthly_weather_2006_2025.csv


[predictions] Loaded cached weather forecast for rainfall_mm_per_day: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/sybilion_gedo/rainfall_mm_per_day_forecast.json


[predictions] Loaded cached weather forecast for temperature_avg_c: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/sybilion_gedo/temperature_avg_c_forecast.json


...

[predictions] Extended fuel series with seasonal monthly median: 2026-05-01 to 2026-05-01.


[predictions] Loaded global fuel price proxy from WFP Somalia fuel rows: 2014-10-01 to 2026-05-01, 140 rows, aggregation=median, missing 6->0.


[predictions] Building local seasonal forecast: global_fuel_sybilion


[predictions] Saved prediction plot: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/somalia/data/sybilion_fuel/global_fuel_sybilion_forecast.png


weather rows: 240 {'rainfall_mm_per_day_mean': 0.9663858091275, 'temperature_avg_c_mean': 28.463808087500002, 'relative_humidity_pct_mean': 50.60598368750001}
water rows: 185 forecast rows: 6
food rows: 184 forecast rows: 6
fuel rows: 140 forecast rows: 6
        date   forecast        q05        q10        q25        q50  \
0 2026-06-01  41.478636  37.330773  37.330773  39.404705  41.478636   
1 2026-07-01  42.973182  38.675864  38.675864  40.824523  42.973182   
2 2026-08-01  43.355000  39.019500  39.019500  41.187250  43.355000   
3 2026-09-01  42.502727  38.252455  38.252455  40.377591  42.502727   
4 2026-10-01  41.005417  36.904875  36.904875  38.955146  41.005417   

         q75        q90        q95  
0  43.552568  45.626500  45.626500  
1  45.121841  47.270500  47.270500  
2  45.522750  47.690500  47.690500  
3  44.627864  46.753000  46.753000  
4  43.055688  45.105958  45.105958  


## Test 2: On-the-fly weather data fetch with cached Sybilion forecasts

This exercises the Copernicus loader path. Sybilion is still cached here so that running the notebook does not accidentally submit multiple API jobs.

In [ ]:
weather_fetched = predict_somalia_weather(data_source='live', forecast_source='cache')

assert len(weather_fetched.history) > 0
assert all(len(frame) > 0 for frame in weather_fetched.forecasts.values())

print('fetched weather rows:', len(weather_fetched.history))
print('weather path:', weather_fetched.weather_path)
print('used range:', weather_fetched.used_range)


[predictions] Gedo polygon: [(42.8903167, 4.3140417), (43.0996111, 3.0833333), (41.5229722, 1.2651944), (40.9924083, 2.7923778), (42.0586444, 4.1903472), (42.8903167, 4.3140417)]


[predictions] CDS bbox [north, west, south, east]: [4.414041699999999, 40.8924083, 1.1651943999999999, 43.1996111]


[predictions] Requesting Gedo weather data from Copernicus: 2006-2025


[predictions] Saved weather data: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/gedo_monthly_weather_2006_2025.csv


[predictions] Loaded cached weather forecast for rainfall_mm_per_day: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/sybilion_gedo/rainfall_mm_per_day_forecast.json


[predictions] Loaded cached weather forecast for temperature_avg_c: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/sybilion_gedo/temperature_avg_c_forecast.json


[predictions] Loaded cached weather forecast for relative_humidity_pct: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/sybilion_gedo/relative_humidity_pct_forecast.json


fetched weather rows: 240
weather path: /home/jjonkhans/HACKATHONS/ZERO_ONE_HACK/project/data/weather/gedo_monthly_weather_2006_2025.csv
used range: (2006, 2025)


## Optional: Live Sybilion forecasts

Set `RUN_LIVE_SYBILION = True` only when you intentionally want to submit fresh Sybilion forecast jobs.

In [ ]:
RUN_LIVE_SYBILION = False

if RUN_LIVE_SYBILION:
    weather_live = predict_somalia_weather(data_source='live', forecast_source='live')
    water_live = predict_somalia_water_prices(source='live')
    food_live = predict_somalia_food_prices(source='live')
    fuel_live = predict_global_fuel_price(source='live')
    print('live weather jobs:', weather_live.job_ids)
    print('live water job:', water_live.job_id)
    print('live food job:', food_live.job_id)
    print('live fuel job:', fuel_live.job_id)
else:
    print('Skipped live Sybilion submissions.')


Skipped live Sybilion submissions.
